# expdpy — Learn panel data

_Notebook version: built 2026-06-25 04:54 UTC — re-open this notebook from GitHub if yours is older, to get the latest version._

A cloud-runnable walkthrough of the **Learn** module of [expdpy](https://github.com/cmg777/expdpy) on the bundled `kuznets` panel. Run the install cell below first, then run the rest top to bottom.

> The first cell installs everything and then **restarts the Colab runtime once** so the upgraded NumPy loads cleanly. When it reconnects, run the cells again (Runtime > Run all) — the install cell skips the restart the second time.

This notebook mirrors the [Learn page](https://cmg777.github.io/expdpy/learn.html) of the docs.

In [ ]:
import importlib.util
import os

!pip install -q "numpy>=2.1" "numba>=0.61" "expdpy @ git+https://github.com/cmg777/expdpy.git"
!pip install -q --force-reinstall --no-deps "expdpy @ git+https://github.com/cmg777/expdpy.git"

_RESTART_FLAG = "/tmp/.expdpy_runtime_restarted"
_ON_COLAB = importlib.util.find_spec("google.colab") is not None
if _ON_COLAB and not os.path.exists(_RESTART_FLAG):
    with open(_RESTART_FLAG, "w"):
        pass
    print("Install complete - restarting the runtime once so NumPy loads cleanly.")
    print("After it reconnects, run the cells again (Runtime > Run all).")
    os.kill(os.getpid(), 9)

In [ ]:
# Ensure Plotly figures render in Google Colab (a no-op in other notebook frontends).
import plotly.io as pio

try:
    import google.colab  # noqa: F401  (present only on Colab)

    pio.renderers.default = "colab"
except ImportError:
    pass

The **Learn** module is the teaching layer behind the rest of the library. The [Explore](explore.qmd)
and [Analyze](analyze.qmd) case studies leaned on a handful of ideas — within vs between variation,
fixed effects, clustered standard errors, convergence, the Kuznets wave. This page explains *why*
those moves work, **two complementary ways**:

1. **A plain-language layer on every result.** Each `explore_*` / `analyze_*` result can explain
   itself with `.interpret()` (an associational reading) and `.explain()` (a concept explainer).
2. **Simulated sandboxes.** Each `learn_*` function generates its *own* data from known parameters,
   so you can see an estimator hit — or miss — a truth you control, and turn the knobs.

Read it top to bottom: we start from the real Kuznets model you fit in Analyze, browse the concept
index, then isolate each idea in a sandbox — the mechanics of fixed effects, two inference classics,
convergence, and the Kuznets wave itself.

::: {.callout-note}
The sandboxes *simulate* data to make a teaching point; the `.interpret()` text is always
**associational** (never a causal claim). The `correlation_vs_causation` explainer spells out what a
causal reading would additionally require.
:::

In [ ]:
import expdpy as ex
from expdpy.data import load_kuznets, load_kuznets_data_def

df = ex.set_labels(load_kuznets(), load_kuznets_data_def(), set_panel=True)

## Stage 1 — Read a real result in plain language

Fit a model anywhere in [Analyze](analyze.qmd), then ask it to explain itself. Here is the two-way
fixed-effects cubic Kuznets curve from the Analyze case study. `.interpret()` gives an
**associational** reading of what was estimated:

In [ ]:
res = ex.analyze_regression_table(
    df,
    dvs="gini_regional",
    idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"],
    feffects=["country", "year"],
    clusters=["country"],
)
print(res.interpret())

`.explain()` returns the **concept explainer** for the method — what it is, when to use it, and its
caveats — the same content as `ex.explain("fixed_effects")`:

In [ ]:
print(res.explain().to_markdown())

Every `explore_*` and `analyze_*` result carries these two methods. The rest of this page is the
ideas they describe, isolated one at a time.

## Stage 2 — The browsable concept index

`ex.list_topics()` returns every registered concept explainer (currently 27); pass any of them (or
an alias) to `ex.explain(...)`:

In [ ]:
ex.list_topics()

In [ ]:
ex.explain("fixed_effects")

The full catalog, grouped by theme — every entry is a key you can pass to `ex.explain(...)`:

| Theme | Explainer topics |
|---|---|
| OLS & regression | `ols`, `fwl`, `omitted_variable_bias`, `descriptive_stats` |
| Panel structure & variation | `panel_structure`, `within_between_variation`, `time_trends`, `transition_matrix` |
| Fixed effects & the within transform | `fixed_effects`, `within_transformation`, `dummy_variables`, `first_differences` |
| Random effects, CRE & Hausman | `random_effects`, `correlated_random_effects`, `hausman` |
| Standard errors & inference | `clustered_se` |
| Convergence | `beta_convergence`, `sigma_convergence`, `convergence_clubs`, `kuznets_waves` |
| Causal designs / DiD | `event_study`, `parallel_trends`, `correlation_vs_causation` |
| Correlation | `pearson`, `spearman` |
| Outlier treatment | `winsorize`, `truncate` |

## Stage 3 — The core identity: first differences ≈ demeaning ≈ dummy variables

A unit fixed effect is anything constant within a unit over time. Three transformations all
**remove** it and recover the same within-unit slope:

- **First differences** subtract each unit's previous-period value (Δy on Δx).
- **The within transformation (demeaning)** subtracts each unit's time-average.
- **Least-squares dummy variables (LSDV)** add one dummy per unit to an OLS regression.

`learn_first_differences` simulates a panel where the regressor is correlated with the unit effect
(so pooled OLS is biased) and recovers the slope by first differences and by demeaning. On a
two-period panel the two coincide exactly, and both recover the truth:

In [ ]:
fd = ex.learn_first_differences(n_periods=2)
print(fd.interpret())
fd.fig

`learn_within_vs_lsdv` shows that demeaning and unit dummies give the **identical** slope — the
Frisch–Waugh–Lovell theorem at work — for any number of periods:

In [ ]:
ex.learn_within_vs_lsdv(n_periods=6).fig

## Stage 4 — Why fixed effects matter

`learn_pooled_vs_fixed_effects` makes the bias concrete: when the unit effect is correlated with the
regressor, pooled OLS is biased, and using only within-unit variation (fixed effects) recovers the
true slope. This is exactly the move the Analyze case study made on the Kuznets curve.

In [ ]:
pf = ex.learn_pooled_vs_fixed_effects(unit_effect_corr=0.8)
print(pf.interpret())
pf.fig

The **correlated random effects (Mundlak)** estimator bridges fixed and random effects — see its
explainer, and `analyze_cre_table` in [Analyze](analyze.qmd):

In [ ]:
ex.explain("correlated_random_effects")

## Stage 5 — Two inference classics

**Omitted-variable bias** — what happens to a coefficient when a correlated confounder is left out.
The short regression is biased; controlling for the confounder recovers the truth:

In [ ]:
ex.learn_omitted_variable_bias(corr_xz=0.6).fig

**Clustered standard errors** — clustering changes the *standard error*, not the point estimate.
Ignoring within-cluster correlation understates uncertainty (the bars shrink too far):

In [ ]:
ex.learn_clustering_se(icc=0.3).fig

## Stage 6 — Convergence, simulated

The Analyze case study asked whether incomes converge. These sandboxes plant a *known* answer so you
can watch each tool recover it.

`learn_beta_convergence` — absolute vs **conditional** convergence on a known-parameter AR(1) panel:
omitting a steady-state determinant biases the unconditional slope; conditioning on it recovers the
true convergence speed. (See [`analyze_beta_convergence`](analyze_beta_convergence.qmd) on real data.)

In [ ]:
bc = ex.learn_beta_convergence(rho=0.9, gamma=0.6, corr=0.7)
print(bc.interpret())
bc.fig

`learn_sigma_convergence` — a panel whose cross-sectional dispersion narrows at a **known** rate:
the standard deviation, the Gini index and the coefficient of variation all shrink at the log-rate
`ln(rho)`, and the function recovers it. (See
[`analyze_sigma_convergence`](analyze_sigma_convergence.qmd) on real data.)

In [ ]:
ex.learn_sigma_convergence(rho=0.93).fig

`learn_convergence_clubs` — a panel with a **planted** club structure: each unit belongs to one of
several clubs converging to distinct levels, so the panel does not converge globally, yet the
Phillips–Sul clustering algorithm recovers the clubs without being told they exist. (See
[`analyze_convergence_clubs`](analyze_convergence_clubs.qmd) on real data.)

In [ ]:
ex.learn_convergence_clubs().fig

## Stage 7 — The Kuznets wave, simulated

`learn_kuznets_waves` is the teaching counterpart to the flagship `analyze_kuznets_waves`. It plants
a **known** polynomial wave into a panel and fits it under three estimators. The within and pooled
estimators recover the true top-order coefficient; the **between** estimator differs — because the
average of a nonlinear function is not the function of the average. That gap is exactly why the
Analyze case study asked whether the N-shape was a within- or a between-country pattern.

In [ ]:
kw = ex.learn_kuznets_waves()
print(kw.interpret())
kw.fig

## Where to go next

- [Explore panel data](explore.qmd) — describe your data before you model it.
- [Analyze panel data](analyze.qmd) — the estimators these ideas underpin, on the real Kuznets panel.
- Browse every concept explainer in the [API reference](reference/index.qmd), or pass any
  `list_topics()` key to `ex.explain(...)`.
- Prefer no code? Launch the [Learn app](streamlit.qmd) — the sandboxes and explainers, interactive.